# 05 Final Load Prep

Use this notebook to compute final KPIs, prepare the Tableau-ready dataset, and export the exact file used in the dashboard.

In [2]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()

## 1. Load Cleaned Dataset

Loading from `data/processed/cleaned_dataset.csv` — the output of Notebook 2.
Verifying shape and key columns before any computation begins.

In [3]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()

DATA_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'
TABLEAU_READY_PATH = PROJECT_ROOT / 'data/processed/tableau_ready_dataset.csv'

df = pd.read_csv(DATA_PATH)

# Restore correct dtypes
df['fl_date'] = pd.to_datetime(df['fl_date'])

df_all = df.copy()
df_ops = df[df['cancelled'] == 0].copy()

print(f"Cleaned dataset loaded successfully")
print(f"Total rows        : {len(df_all):,}")
print(f"Operational rows  : {len(df_ops):,}")
print(f"Cancelled rows    : {len(df_all) - len(df_ops):,}")
print(f"Columns           : {df_all.shape[1]}")
print(f"Date range        : {df_all['fl_date'].min().date()} → {df_all['fl_date'].max().date()}")
print(f"Airlines          : {df_all['airline'].nunique()}")
print(f"Unique routes     : {df_all['route'].nunique():,}")

Cleaned dataset loaded successfully
Total rows        : 299,532
Operational rows  : 291,665
Cancelled rows    : 7,867
Columns           : 39
Date range        : 2019-01-01 → 2023-08-31
Airlines          : 18
Unique routes     : 7,101


## 2. KPI Computation — Delay Metrics

Computing all headline delay KPIs. These are the numbers that appear on the
executive summary view of the Tableau dashboard. Every figure is computed from
`df_ops` — operational flights only, consistent with Notebooks 3 and 4.

In [4]:
# Core delay KPIs
total_flights     = len(df_all)
total_ops_flights = len(df_ops)
total_delayed     = df_ops['is_delayed'].sum()
delay_rate        = total_delayed / total_ops_flights * 100

avg_arr_delay_all    = df_ops['arr_delay'].mean()
avg_arr_delay_delayed = df_ops[df_ops['is_delayed'] == True]['arr_delay'].mean()

on_time_rate = 100 - delay_rate

severe_delay_rate = (df_ops['delay_bucket'] == '60+ min').sum() / total_ops_flights * 100
moderate_delay_rate = (df_ops['delay_bucket'] == '15-60 min').sum() / total_ops_flights * 100

print("=" * 55)
print("DELAY KPIs")
print("=" * 55)
print(f"Total scheduled flights       : {total_flights:,}")
print(f"Total operational flights     : {total_ops_flights:,}")
print(f"Total delayed flights         : {total_delayed:,}")
print(f"On-Time Rate                  : {on_time_rate:.2f}%")
print(f"Delay Rate (>15 min)          : {delay_rate:.2f}%")
print(f"Moderate Delay Rate (15-60min): {moderate_delay_rate:.2f}%")
print(f"Severe Delay Rate (60+ min)   : {severe_delay_rate:.2f}%")
print(f"Avg Arrival Delay (all ops)   : {avg_arr_delay_all:.2f} mins")
print(f"Avg Arrival Delay (delayed)   : {avg_arr_delay_delayed:.2f} mins")
print("=" * 55)

DELAY KPIs
Total scheduled flights       : 299,532
Total operational flights     : 291,665
Total delayed flights         : 51,203
On-Time Rate                  : 82.44%
Delay Rate (>15 min)          : 17.56%
Moderate Delay Rate (15-60min): 11.68%
Severe Delay Rate (60+ min)   : 5.87%
Avg Arrival Delay (all ops)   : 3.15 mins
Avg Arrival Delay (delayed)   : 63.49 mins


## 3. KPI Computation — Cancellation Metrics

Cancellation KPIs use `df_all` — the full scheduled flight dataset including
cancelled records. The cause breakdown is computed from cancelled flights only.

In [5]:
total_cancelled   = df_all['cancelled'].sum()
cancel_rate       = total_cancelled / total_flights * 100

mapping = {'A': 'Carrier', 'B': 'Weather', 'C': 'NAS', 'D': 'Security'}
df_cancelled = df_all[df_all['cancelled'] == 1].copy()
cancel_by_cause = (
    df_cancelled['cancellation_code']
    .map(mapping)
    .value_counts(normalize=True) * 100
).round(2)

print("=" * 55)
print("CANCELLATION KPIs")
print("=" * 55)
print(f"Total cancelled flights  : {total_cancelled:,}")
print(f"Cancellation Rate        : {cancel_rate:.2f}%")
print()
print("Cancellation by Cause:")
for cause, pct in cancel_by_cause.items():
    print(f"  {cause:<15}: {pct:.2f}%")
print("=" * 55)

CANCELLATION KPIs
Total cancelled flights  : 7,867
Cancellation Rate        : 2.63%

Cancellation by Cause:
  Weather        : 36.16%
  Security       : 31.40%
  Carrier        : 24.85%
  NAS            : 7.59%


## 4. KPI Computation — Airline Performance

Computing delay rate and cancellation rate per airline. These feed the airline
performance view on the Tableau dashboard. Delay rate uses `df_ops`,
cancellation rate uses `df_all`.

In [6]:
airline_kpis = pd.DataFrame({
    'total_flights'    : df_all.groupby('airline')['cancelled'].count(),
    'cancelled_flights': df_all.groupby('airline')['cancelled'].sum(),
    'cancel_rate_pct'  : df_all.groupby('airline')['cancelled'].mean() * 100,
    'delayed_flights'  : df_ops.groupby('airline')['is_delayed'].sum(),
    'ops_flights'      : df_ops.groupby('airline')['is_delayed'].count(),
    'delay_rate_pct'   : df_ops.groupby('airline')['is_delayed'].mean() * 100,
    'avg_arr_delay'    : df_ops.groupby('airline')['arr_delay'].mean(),
    'avg_dep_delay'    : df_ops.groupby('airline')['dep_delay'].mean(),
}).round(2)

airline_kpis['on_time_rate_pct'] = 100 - airline_kpis['delay_rate_pct']

print("Airline KPI Table:")
print(airline_kpis.sort_values('delay_rate_pct', ascending=False).to_string())

Airline KPI Table:
                                    total_flights  cancelled_flights  cancel_rate_pct  delayed_flights  ops_flights  delay_rate_pct  avg_arr_delay  avg_dep_delay  on_time_rate_pct
airline                                                                                                                                                                            
JetBlue Airways                             11361                306             2.69             2900        11055           26.23          11.24          17.35             73.77
Frontier Airlines Inc.                       6284                154             2.45             1599         6130           26.08           9.85          14.69             73.92
Allegiant Air                                5181                247             4.77             1179         4934           23.90          10.12          10.85             76.10
Spirit Air Lines                             9640                211             

## 5. KPI Computation — Route Performance

Computing delay and cancellation metrics per route. Only routes with more than
100 total scheduled flights are included — low-frequency routes would produce
unreliable rates from small samples.

In [7]:
# Filter to routes with sufficient volume
route_volume = df_all.groupby('route')['cancelled'].count()
valid_routes = route_volume[route_volume >= 100].index

df_routes_ops = df_ops[df_ops['route'].isin(valid_routes)]
df_routes_all = df_all[df_all['route'].isin(valid_routes)]

route_kpis = pd.DataFrame({
    'total_flights'    : df_routes_all.groupby('route')['cancelled'].count(),
    'cancel_rate_pct'  : df_routes_all.groupby('route')['cancelled'].mean() * 100,
    'delay_rate_pct'   : df_routes_ops.groupby('route')['is_delayed'].mean() * 100,
    'avg_arr_delay'    : df_routes_ops.groupby('route')['arr_delay'].mean(),
    'avg_distance'     : df_routes_ops.groupby('route')['distance'].mean(),
}).round(2)

print(f"Routes with 100+ flights: {len(route_kpis):,}")
print()
print("Top 10 Routes by Delay Rate:")
print(route_kpis.sort_values('delay_rate_pct', ascending=False).head(10).to_string())
print()
print("Top 10 Routes by Cancellation Rate:")
print(route_kpis.sort_values('cancel_rate_pct', ascending=False).head(10).to_string())

Routes with 100+ flights: 834

Top 10 Routes by Delay Rate:
           total_flights  cancel_rate_pct  delay_rate_pct  avg_arr_delay  avg_distance
route                                                                                 
SJU → EWR            101             0.00           34.65          18.20        1608.0
ORD → EWR            184             6.52           33.14          17.66         719.0
IAH → EWR            153             2.61           31.54          18.46        1400.0
EWR → SFO            181             2.76           31.25          12.16        2565.0
MCO → EWR            292             2.74           30.99          14.07         937.0
FLL → EWR            237             2.95           30.87          11.04        1065.0
FLL → ORD            125             4.00           30.83          11.37        1182.0
DEN → EWR            155             1.29           30.72          12.26        1605.0
FLL → JFK            187             1.60           30.43          22.

## 6. KPI Computation — Time-Based Metrics

Computing monthly and yearly delay and cancellation trends. These feed the
time-series views on the Tableau dashboard.

In [8]:
# Monthly KPIs
monthly_kpis = pd.DataFrame({
    'delay_rate_pct'  : df_ops.groupby('month')['is_delayed'].mean() * 100,
    'avg_arr_delay'   : df_ops.groupby('month')['arr_delay'].mean(),
    'cancel_rate_pct' : df_all.groupby('month')['cancelled'].mean() * 100,
    'total_flights'   : df_all.groupby('month')['cancelled'].count(),
}).round(2)

month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
monthly_kpis.index = monthly_kpis.index.map(month_map)

print("Monthly KPIs:")
print(monthly_kpis.to_string())
print()

# Yearly KPIs
yearly_kpis = pd.DataFrame({
    'delay_rate_pct'  : df_ops.groupby('year')['is_delayed'].mean() * 100,
    'avg_arr_delay'   : df_ops.groupby('year')['arr_delay'].mean(),
    'cancel_rate_pct' : df_all.groupby('year')['cancelled'].mean() * 100,
    'total_flights'   : df_all.groupby('year')['cancelled'].count(),
}).round(2)

print("Yearly KPIs:")
print(yearly_kpis.to_string())

# Day of week KPIs
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_kpis = pd.DataFrame({
    'delay_rate_pct'  : df_ops.groupby('day_of_week')['is_delayed'].mean() * 100,
    'avg_arr_delay'   : df_ops.groupby('day_of_week')['arr_delay'].mean(),
    'cancel_rate_pct' : df_all.groupby('day_of_week')['cancelled'].mean() * 100,
}).reindex(day_order).round(2)

print()
print("Day of Week KPIs:")
print(dow_kpis.to_string())

Monthly KPIs:
       delay_rate_pct  avg_arr_delay  cancel_rate_pct  total_flights
month                                                               
Jan             15.85           0.89             2.71          26902
Feb             17.82           2.57             2.94          24989
Mar             16.13           1.12             5.01          29172
Apr             17.09           2.29             6.93          25356
May             17.03           2.60             1.75          25114
Jun             22.55           8.66             1.83          26130
Jul             21.48           7.84             1.77          28358
Aug             19.34           5.72             1.95          28735
Sep             12.77          -1.09             1.20          20731
Oct             14.68           0.37             1.05          21795
Nov             13.95          -0.67             0.86          20940
Dec             19.91           5.36             2.57          21310

Yearly KPIs:
      

## 7. KPI Computation — Delay Cause Breakdown

Computing total delay minutes and percentage share per cause. These feed the
delay cause breakdown chart on the Tableau dashboard.

In [9]:
delay_cause_cols = {
    'delay_due_carrier'       : 'Carrier',
    'delay_due_weather'       : 'Weather',
    'delay_due_nas'           : 'NAS',
    'delay_due_security'      : 'Security',
    'delay_due_late_aircraft' : 'Late Aircraft'
}

cause_totals = df_ops[list(delay_cause_cols.keys())].sum()
cause_totals.index = list(delay_cause_cols.values())
cause_pct = (cause_totals / cause_totals.sum() * 100).round(2)

cause_kpis = pd.DataFrame({
    'total_delay_minutes': cause_totals,
    'share_pct'          : cause_pct
})

controllable = ['Carrier', 'Late Aircraft']
cause_kpis['controllable'] = cause_kpis.index.isin(controllable)

controllable_pct = cause_kpis[cause_kpis['controllable']]['share_pct'].sum()
external_pct     = cause_kpis[~cause_kpis['controllable']]['share_pct'].sum()

print("Delay Cause KPI Table:")
print(cause_kpis.to_string())
print()
print(f"Controllable share (Carrier + Late Aircraft) : {controllable_pct:.1f}%")
print(f"External share     (Weather + NAS + Security): {external_pct:.1f}%")

Delay Cause KPI Table:
               total_delay_minutes  share_pct  controllable
Carrier                  1131408.0      34.51          True
Weather                   180492.0       5.51         False
NAS                       678033.0      20.68         False
Security                    7586.0       0.23         False
Late Aircraft            1281170.0      39.08          True

Controllable share (Carrier + Late Aircraft) : 73.6%
External share     (Weather + NAS + Security): 26.4%


## 8. Tableau Dataset Preparation

The Tableau-ready export is a refined version of the cleaned dataset with:
- All engineered features retained
- Column names converted to clean, readable labels
- Redundant identifier columns dropped (airline_dot, dot_code — duplicates
  of airline and airline_code)
- Wheel times and taxi times dropped — not needed for dashboard KPIs
- A `delay_controllable` flag added to enable Tableau filtering by
  controllable vs external delay causes
- All remaining columns verified for correct data types

This is the exact file loaded into Tableau Public — no further transformation
happens inside Tableau itself.

In [10]:
tableau_df = df_all.copy()

# Drop columns not needed for dashboard
drop_cols = [
    'airline_dot',      # Duplicate of airline + airline_code combined
    'dot_code',         # Internal BTS identifier — not useful for dashboard
    'wheels_off',       # Operational detail not used in any KPI
    'wheels_on',        # Operational detail not used in any KPI
    'taxi_out',         # Operational detail not used in any KPI
    'taxi_in',          # Operational detail not used in any KPI
    'elapsed_time',     # Redundant with air_time + taxi times
    'fl_number',        # Flight number — too granular for dashboard
]

# Only drop columns that exist
drop_cols = [c for c in drop_cols if c in tableau_df.columns]
tableau_df = tableau_df.drop(columns=drop_cols)

# Add controllable delay flag at flight level
# True if any controllable cause (carrier or late aircraft) contributed
tableau_df['delay_controllable'] = (
    (tableau_df['delay_due_carrier'] > 0) |
    (tableau_df['delay_due_late_aircraft'] > 0)
)

# Add total controllable and external delay minutes per flight
tableau_df['controllable_delay_mins'] = (
    (tableau_df['delay_due_carrier'] if 'delay_due_carrier' in tableau_df.columns else 0) +
    (tableau_df['delay_due_late_aircraft'] if 'delay_due_late_aircraft' in tableau_df.columns else 0)
)
tableau_df['external_delay_mins'] = (
    (tableau_df['delay_due_weather'] if 'delay_due_weather' in tableau_df.columns else 0) +
    (tableau_df['delay_due_nas'] if 'delay_due_nas' in tableau_df.columns else 0) +
    (tableau_df['delay_due_security'] if 'delay_due_security' in tableau_df.columns else 0)
)

# Clean month name column for Tableau labels
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
tableau_df['month_name'] = tableau_df['month'].map(month_map)

# Verify final shape
print(f"Tableau-ready dataset:")
print(f"  Rows    : {len(tableau_df):,}")
print(f"  Columns : {tableau_df.shape[1]}")
print()
print("Columns in export:")
for col in tableau_df.columns:
    print(f"  {col}")

# Final null check
remaining_nulls = tableau_df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print()
if remaining_nulls.empty:
    print("✓ No unexpected nulls in export dataset")
else:
    print("Remaining nulls (intentional — cancelled flights):")
    print(remaining_nulls.to_string())

Tableau-ready dataset:
  Rows    : 299,532
  Columns : 35

Columns in export:
  fl_date
  airline
  airline_code
  origin
  origin_city
  dest
  dest_city
  crs_dep_time
  dep_time
  dep_delay
  crs_arr_time
  arr_time
  arr_delay
  cancelled
  cancellation_code
  diverted
  crs_elapsed_time
  air_time
  distance
  delay_due_carrier
  delay_due_weather
  delay_due_nas
  delay_due_security
  delay_due_late_aircraft
  is_delayed
  delay_bucket
  month
  day_of_week
  year
  route
  total_delay_cause
  delay_controllable
  controllable_delay_mins
  external_delay_mins
  month_name

Remaining nulls (intentional — cancelled flights):
dep_time            7708
dep_delay           7708
arr_time            7951
arr_delay           7867
crs_elapsed_time       1
air_time            8614


## 9. Final Export

Exporting the Tableau-ready dataset to `data/processed/tableau_ready.csv`.
This is the file that gets loaded directly into Tableau Public.

After export, a final verification read confirms the file wrote correctly
and the row and column counts match what was prepared above.

In [11]:
TABLEAU_READY_PATH.parent.mkdir(parents=True, exist_ok=True)
tableau_df.to_csv(TABLEAU_READY_PATH, index=False)

# Verification read
verify_df = pd.read_csv(TABLEAU_READY_PATH)
print(f"✓ Export verified successfully")
print(f"  File     : {TABLEAU_READY_PATH}")
print(f"  Rows     : {len(verify_df):,}")
print(f"  Columns  : {verify_df.shape[1]}")
print()
print(f"  File size: {TABLEAU_READY_PATH.stat().st_size / 1024 / 1024:.2f} MB")
assert len(verify_df) == len(tableau_df), "Row count mismatch — export failed"
assert verify_df.shape[1] == tableau_df.shape[1], "Column count mismatch — export failed"
print()
print("✓ All assertions passed — tableau_ready_dateset.csv is consistent with prepared dataset")

✓ Export verified successfully
  File     : /Users/preetishubhrani/Documents/Projects/SectionB_G-18_Flight_Delay_And_Cancellation_Analysis/data/processed/tableau_ready_dataset.csv
  Rows     : 299,532
  Columns  : 35

  File size: 64.98 MB

✓ All assertions passed — tableau_ready_dateset.csv is consistent with prepared dataset


## 10. Summary — What This Notebook Produced

| Output | Details |
|--------|---------|
| File exported | `data/processed/tableau_ready_dataset.csv` |
| Rows | ~291,665 (verified above) |
| Columns | Cleaned + engineered + Tableau-specific flags |
| KPIs computed | Delay rate, cancellation rate, airline rankings, route rankings, time trends, cause breakdown |
| New columns added | `delay_controllable`, `controllable_delay_mins`, `external_delay_mins`, `month_name` |
| Columns removed | Redundant identifiers and operational timing fields not used in dashboard |
